## baseline:

In [5]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.model import CNNBaseline
from src.train import evaluate, fit, make_scheduler, train_one_epoch
from src.utils import get_device, set_seed


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
set_seed(SEED)
device = get_device()

IMG_SIZE = 128
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder('../data/processed/train', transform=base_transform)
val_ds = datasets.ImageFolder('../data/processed/val', transform=base_transform)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
device= get_device()
x , y = next(iter(train_loader))
print(x.shape , y.shape)
model= CNNBaseline().to(device)
print(model(x.to(device)).shape)
print(sum(p.numel() for p in model.parameters()))

torch.Size([32, 3, 128, 128]) torch.Size([32])
torch.Size([32, 8])
94280


In [6]:
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
m = CNNBaseline().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
for step in range(100):
    opt.zero_grad()
    loss = crit(m(x), y)
    loss.backward()
    opt.step()
    if step % 20 == 0:
        print(step, loss.item())

0 2.0943050384521484
20 1.6087497472763062
40 1.1030491590499878
60 0.693679928779602
80 0.3674524426460266


## training loop :

In [ ]:
EPOCHS = 20
model = CNNBaseline().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr=1e-3)
meta = {
    'architecture': 'CNNBaseline',
    'class_to_idx': train_ds.class_to_idx ,
    'IMG_SIZE' : IMG_SIZE,
    'seed': SEED,
    'config': { 'epochs': EPOCHS , 'lr': 1e-3 , 'batch_size': 32 , 'optimizer':'Adam'}}
history = fit(model , train_loader , val_loader , optimizer , criterion , device , EPOCHS , '../checkpoints/baseline.pt' , '../logs/baseline.csv' , meta = meta)

{'epoch': 1, 'train_loss': 1.980867289272975, 'val_loss': 1.8716043862429532, 'val_macro_f1': 0.2423718009797562, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.828980234553862, 'val_loss': 1.8233513209726904, 'val_macro_f1': 0.2403213410795215, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.7798448207705264, 'val_loss': 1.7686392616916013, 'val_macro_f1': 0.2798345488028936, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.724175269376, 'val_loss': 1.7621663976025272, 'val_macro_f1': 0.32276564048712597, 'lr': 0.001}
{'epoch': 5, 'train_loss': 1.6514100493975783, 'val_loss': 1.6932384580760806, 'val_macro_f1': 0.2795967205711146, 'lr': 0.001}
{'epoch': 6, 'train_loss': 1.5955122349090278, 'val_loss': 1.5686763738656972, 'val_macro_f1': 0.3546851728599807, 'lr': 0.001}
{'epoch': 7, 'train_loss': 1.4976062622838706, 'val_loss': 1.5174716788452942, 'val_macro_f1': 0.36182763438881765, 'lr': 0.001}
{'epoch': 8, 'train_loss': 1.4430010885755453, 'val_loss': 1.4536306963338481, 'val_macro_f1': 0.43670200532